# 可信心率观测系统
公开LIVE入口使用完整发布清单验证运行资产，不携带私人生理实录，也不冒充历史实录资格已通过。

请先阅读 `README_PUBLIC_LIVE.md`，使用对应PYNQ-Z2系统与接线。依次执行四个代码单元，不使用Run All。第二单元将启动20秒视频检查，第四单元将实际采集；都需要手动输入确认。

停止时点击第三单元输出中的“正常停止并导出”，等待第四单元返回；不要用中断内核代替正常停止。失败或已有attempt文件时保留目录，不自动重跑。

In [ ]:
from pathlib import Path
import hashlib, os, re, sys
sys.dont_write_bytecode = True
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
release_root = Path.cwd().resolve()
anchor = (release_root / 'public-live-manifest.sha256').read_text('ascii').split()[0]
if re.fullmatch(r'[0-9a-f]{64}', anchor) is None:
    raise ValueError('发布清单SHA256格式错误')
if hashlib.sha256((release_root / 'public-live-manifest.json').read_bytes()).hexdigest() != anchor:
    raise ValueError('发布清单与独立SHA256锚不符')
print('请先把该SHA256与发布页独立校验值核对：', anchor)
runtime_root = release_root / 'runtime'
sys.path.insert(0, str(runtime_root))
from public_live_entry import PublicLiveSession
session = PublicLiveSession(release_root, anchor)
checked = session.check()
print(checked)
if checked['dependency_complete'] is not True:
    raise RuntimeError('目标依赖或CPU ABI检查未通过；请按README补齐匹配板端环境后再继续。未写短测attempt、未启动原生后端。')
report = session.short()
print(report)


In [ ]:
confirmation = input('其他硬件入口已正常收尾。启动20秒视频资格检查，输入 RUN-PUBLIC-VIDEO20：')
report = session.preview20(hardware=True, confirmation=confirmation)
print(report)


In [ ]:
report = session.prepare()
print(report)
# 保持本单元的绘图和“正常停止并导出”按钮可见，等待绿色就绪提示。


In [ ]:
confirmation = input('已看到视频正常收尾、绘图绿色就绪，手指放稳。输入 RUN-PUBLIC-LIVE25：')
report = session.run(hardware=True, confirmation=confirmation)
print(report)
from IPython.display import FileLink, display
display(FileLink('runtime/continuous-results.zip'))
